In [1]:
# NOTEBOOK 04: CONCERN TYPE AND FUNCTION TAGGING

import os

import pandas as pd


In [2]:
# ----------------------------------------------------------
# STEP 0: SETUP
# ----------------------------------------------------------

QUALITY_PATH     = '../data/quality_check'
RISK_SCORING_DIR = '../data/risk_scoring'
OUTPUT_PATH      = '../data/risk_scoring'
os.makedirs(OUTPUT_PATH, exist_ok=True)

CLASSIFICATION_FILE = os.path.join(QUALITY_PATH,
                                   'classification_top400_COMPLETE.xlsx')
LONG_FILE           = os.path.join(RISK_SCORING_DIR,
                                   'product_ingredient_long.csv')
SCORES_FILE         = os.path.join(RISK_SCORING_DIR,
                                   'product_risk_scores.csv')

print("STEP 0: Setup")
print()
print(f"  {'File':<45} {'Status':>10}")
print(f"  {'----':<45} {'------':>10}")
for f in [CLASSIFICATION_FILE, LONG_FILE, SCORES_FILE]:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f"  {os.path.basename(f):<45} {status:>10}")


STEP 0: Setup

  File                                              Status
  ----                                              ------
  classification_top400_COMPLETE.xlsx                   OK
  product_ingredient_long.csv                           OK
  product_risk_scores.csv                               OK


In [3]:
# ----------------------------------------------------------
# STEP 1: LOAD INPUTS
# ----------------------------------------------------------

print()
print("STEP 1: Load Inputs")
print()

classification = pd.read_excel(CLASSIFICATION_FILE)
long_df        = pd.read_csv(LONG_FILE)
scores         = pd.read_csv(SCORES_FILE)

classified = classification[classification['risk_level'].notna()].copy()

print(f"  {'Classified ingredients':<40} {len(classified):>10,}")
print(f"  {'Product-ingredient rows':<40} {len(long_df):>10,}")
print(f"  {'Products':<40} {len(scores):>10,}")



STEP 1: Load Inputs

  Classified ingredients                          400
  Product-ingredient rows                     227,213
  Products                                      7,544


In [4]:
# ----------------------------------------------------------
# STEP 2: DEFINE CONCERN TAG RULES (15 categories)
# ----------------------------------------------------------

print()
print("STEP 2: Define Concern Tag Rules")
print()

CONCERN_TAG_RULES = {
    'eu_banned': [
        'banned in eu', 'banned in leave-on'
    ],
    'annex_iii_restricted': [
        'concentration limit', 'permitted with concentration',
        'restricted ', 'sccs restricts', 'sccs permitted',
        'restricted active'
    ],
    'fragrance_allergen': [
        'fragrance allergen', 'recognized fragrance allergen',
        'rare contact allergen', 'contact allergen',
        'undisclosed allergen', 'allergen mixture',
        'declaration required'
    ],
    'sensitizer': [
        'sensitize', 'sensitization', 'sensitizer',
        'allergic dermatitis', 'severe skin reaction',
        'allergen of the year', 'contact allergy',
        'allergy', 'sulfite-sensitive'
    ],
    'endocrine_disruptor': [
        'endocrine disruptor', 'endocrine-related',
        'hormone balance', 'hormone', 'edc'
    ],
    'carcinogen_concern': [
        'carcinogen', 'cancer-causing'
    ],
    'reproductive_toxin': [
        'reproductive toxin', 'pregnancy precaution'
    ],
    'formaldehyde_releaser': [
        'formaldehyde-releasing', 'formaldehyde releaser',
        'releases formaldehyde'
    ],
    'nitrosamine_concern': [
        'nitrosamine'
    ],
    'photosensitizer': [
        'phototoxic', 'sun exposure', 'skin burns with sun',
        'sensitivity to sunlight', 'furocoumarin'
    ],
    'microplastic': [
        'microplastic', 'teflon'
    ],
    'manufacturing_impurity': [
        '1,4-dioxane', 'manufacturing impurit',
        'residual impurity'
    ],
    'irritant': [
        'irritate', 'irritation', 'irritant',
        'may cause skin', 'cause skin react'
    ],
    'drying': [
        'drying', 'barrier-disruption', 'barrier disruption'
    ],
    'environmental_concern': [
        'environmental review', 'targeted for phase-out',
        'phase-outs'
    ],
}


print(f"  {'Total concern tag categories':<40} "
      f"{len(CONCERN_TAG_RULES):>10,}")



STEP 2: Define Concern Tag Rules

  Total concern tag categories                     15


In [5]:
# ----------------------------------------------------------
# STEP 3: DEFINE FUNCTION TAG RULES (30 categories)
# ----------------------------------------------------------
# First matching rule wins (priority order)

print()
print("STEP 3: Define Function Tag Rules")
print()

FUNCTION_TAG_RULES = [
    ('uv_filter',           ['uv filter', 'sunscreen']),
    ('preservative',        ['preservative']),
    ('surfactant',          ['surfactant', 'cleanser',
                             'foaming agent', 'cleansing']),
    ('emulsifier',          ['emulsifier', 'emulsifying',
                             'blends oil and water',
                             'helps dissolve oil',
                             'helps dissolve']),
    ('colorant',            ['colorant', 'pigment',
                             'glitter effect', 'pearly appearance',
                             'skin tone enhancer']),
    ('antioxidant',         ['antioxidant']),
    ('thickener',           ['thickener', 'gellant', 'gelling',
                             'rheology', 'gum', 'binder']),
    ('humectant',           ['humectant']),
    ('moisturizer',         ['moisturizer', 'moisturizing',
                             'moisturising', 'moistrurizer']),
    ('emollient',           ['emollient']),
    ('absorbent',           ['absorbent', 'anti-caking', 'anti caking',
                             'moisture absorption',
                             'texture enhancement', 'optical-finish',
                             'absorption', 'absorbs oil',
                             'prevents clumping']),
    ('chelator',            ['chelating agent', 'chelator']),
    ('film_former',         ['film-former', 'film former',
                             'film-forming']),
    ('soothing',            ['soothe', 'soothing', 'calms skin',
                             'calming']),
    ('wax',                 ['wax', 'structurant']),
    ('skin_barrier',        ['skin lipid', 'skin barrier',
                             'barrier function',
                             'natural moisturizing factor',
                             'natural lipid', 'cell membrane lipid',
                             'absorb into skin', 'barrier lipid']),
    ('conditioner',         ['conditioner', 'conditioning',
                             'antistatic', 'strengthens hair']),
    ('amino_acid',          ['amino acid', 'building block',
                             'amino-acid lipid']),
    ('peptide',             ['peptide']),
    ('vitamin',             ['vitamin']),
    ('silicone',            ['silicone']),
    ('fatty_acid',          ['fatty acid', 'fatty ester']),
    ('plant_extract',       ['plant-derived', 'plant extract',
                             'fruit extract', 'leaf extract',
                             'root extract', 'flower oil',
                             'seed oil', 'seed butter',
                             'seed extract', 'kernel oil',
                             'extract', 'almond oil', 'olive oil',
                             'rosehip oil', 'meadowfoam oil',
                             'avocado oil', 'coconut oil',
                             'argan oil', 'rice protein',
                             'corn starch']),
    ('ph_adjuster',         ['ph adjuster', 'ph buffer']),
    ('solvent',             ['solvent']),
    ('penetration_enhancer', ['penetration enhancer']),
    ('exfoliant',           ['exfoliant', 'exfoliating']),
    ('polymer',             ['synthetic polymer', 'synthetic powder']),
    ('mineral',             ['mineral salt', 'mineral;']),
    ('salt',                ['salt; safe']),
]

print(f"  {'Total function tag categories':<40} "
      f"{len(FUNCTION_TAG_RULES):>10,}")



STEP 3: Define Function Tag Rules

  Total function tag categories                    30


In [6]:
# ----------------------------------------------------------
# STEP 4: APPLY TAGS TO EACH CLASSIFIED INGREDIENT
# ----------------------------------------------------------

print()
print("STEP 4: Apply Tags")
print()


def find_concern_tags(text):
    """Return a list of all concern tags matching the text."""
    if not isinstance(text, str):
        return []
    t = text.lower()
    return [tag for tag, keywords in CONCERN_TAG_RULES.items()
            if any(kw in t for kw in keywords)]


def find_function_tag(text):
    """Return the first matching function tag, or 'other'."""
    if not isinstance(text, str):
        return 'other'
    t = text.lower()
    for tag, keywords in FUNCTION_TAG_RULES:
        if any(kw in t for kw in keywords):
            return tag
    return 'other'


classified['concern_tags']     = classified['health_concern'].apply(find_concern_tags)
classified['concern_tag_list'] = classified['concern_tags'].apply(
    lambda lst: '; '.join(lst) if lst else ''
)
classified['n_concern_tags']   = classified['concern_tags'].apply(len)
classified['function_tag']     = classified['health_concern'].apply(find_function_tag)

# Coverage diagnostics
n_with_concern  = (classified['n_concern_tags'] > 0).sum()
n_with_function = (classified['function_tag'] != 'other').sum()
low_medium_or_higher = classified[classified['risk_level'].isin(
    ['Low-Medium', 'Medium', 'Medium-High', 'High']
)]
higher_risk_without_concern = low_medium_or_higher[
    low_medium_or_higher['n_concern_tags'] == 0
]

print(f"  {'Ingredients with >=1 concern tag':<45} "
      f"{n_with_concern:>6,} / {len(classified):,}")
print(f"  {'Ingredients with function tag (not other)':<45} "
      f"{n_with_function:>6,} / {len(classified):,}")

if len(higher_risk_without_concern):
    print()
    print(
        "  Low-Medium or higher ingredients without concern tag: "
        f"{len(higher_risk_without_concern)}"
    )
    print("  (Review these — text may say 'safe' or use unusual phrasing)")
    for _, r in higher_risk_without_concern.iterrows():
        print(f"    [{r['risk_level']}] {r['canonical_name']}: "
              f"{r['health_concern']}")

print()
print("  Concern tag frequencies:")
all_concern_tags = [t for lst in classified['concern_tags'] for t in lst]
for tag, count in pd.Series(all_concern_tags).value_counts().items():
    print(f"    {tag:<30} {count:>4,}")

print()
print("  Function tag frequencies:")
for tag, count in classified['function_tag'].value_counts().items():
    print(f"    {tag:<30} {count:>4,}")



STEP 4: Apply Tags

  Ingredients with >=1 concern tag                  70 / 400
  Ingredients with function tag (not other)        363 / 400

  Low-Medium or higher ingredients without concern tag: 1
  (Review these — text may say 'safe' or use unusual phrasing)
    [Low-Medium] cyclopentasiloxane: Volatile silicone; safe at cosmetic levels

  Concern tag frequencies:
    irritant                         42
    fragrance_allergen               23
    annex_iii_restricted             16
    sensitizer                        9
    endocrine_disruptor               5
    photosensitizer                   4
    drying                            3
    manufacturing_impurity            3
    nitrosamine_concern               3
    eu_banned                         2
    reproductive_toxin                2
    environmental_concern             2
    microplastic                      1
    carcinogen_concern                1

  Function tag frequencies:
    moisturizer                      4

In [7]:
# ----------------------------------------------------------
# STEP 5: BUILD INGREDIENT-LEVEL TAG TABLE
# ----------------------------------------------------------

print()
print("STEP 5: Build Ingredient Tag Table")
print()

ingredient_tags = classified[[
    'rank', 'canonical_name', 'total_mentions',
    'risk_level', 'sub_weight', 'health_concern',
    'function_tag', 'concern_tag_list', 'n_concern_tags',
]].copy()

ingredient_tags = ingredient_tags.sort_values(
    ['sub_weight', 'total_mentions'], ascending=[False, False]
).reset_index(drop=True)

print(f"  {'Rows in ingredient_tags':<40} "
      f"{len(ingredient_tags):>10,}")



STEP 5: Build Ingredient Tag Table

  Rows in ingredient_tags                         400


In [8]:
# ----------------------------------------------------------
# STEP 6: ENRICH LONG TABLE WITH TAGS
# ----------------------------------------------------------

print()
print("STEP 6: Enrich Long Table with Tags")
print()

canon_to_concerns = dict(zip(classified['canonical_name'],
                             classified['concern_tag_list']))
canon_to_function = dict(zip(classified['canonical_name'],
                             classified['function_tag']))

long_tagged = long_df.copy()
long_tagged['concern_tag_list'] = (
    long_tagged['canonical_name'].map(canon_to_concerns).fillna('')
)
long_tagged['function_tag'] = (
    long_tagged['canonical_name'].map(canon_to_function).fillna('')
)

print(f"  {'Rows in long_tagged':<40} "
      f"{len(long_tagged):>10,}")
print(f"  {'Rows with >=1 concern tag':<40} "
      f"{(long_tagged['concern_tag_list'] != '').sum():>10,}")
print(f"  {'Rows with assigned function value':<40} "
      f"{(long_tagged['function_tag'] != '').sum():>10,}")



STEP 6: Enrich Long Table with Tags

  Rows in long_tagged                         227,213
  Rows with >=1 concern tag                    42,066
  Rows with assigned function value           161,110


In [9]:
# ----------------------------------------------------------
# STEP 7: BUILD PER-PRODUCT CONCERN SUMMARY
# ----------------------------------------------------------

print()
print("STEP 7: Build Per-Product Concern Summary")
print()

classified_long = long_tagged[
    long_tagged['risk_level'] != 'Unclassified'
].copy()
classified_long['concern_tags_split'] = (
    classified_long['concern_tag_list'].str.split('; ')
)
exploded = classified_long.explode('concern_tags_split')
exploded = exploded[
    exploded['concern_tags_split'].notna() &
    (exploded['concern_tags_split'] != '')
]

concern_counts_per_product = (
    exploded.groupby(['product_id', 'concern_tags_split'])
            .size()
            .unstack(fill_value=0)
)
concern_counts_per_product.columns = [
    f'n_{c}' for c in concern_counts_per_product.columns
]
concern_counts_per_product = concern_counts_per_product.reset_index()

product_concern = scores[[
    'product_id', 'product_name', 'brand_name',
    'primary_category', 'weighted_score',
    'final_risk_label', 'coverage_pct',
    'has_high_risk_warning'
]].merge(concern_counts_per_product, on='product_id', how='left')

concern_cols = [c for c in product_concern.columns if c.startswith('n_')]
product_concern[concern_cols] = (
    product_concern[concern_cols].fillna(0).astype(int)
)
product_concern['total_concern_flags'] = (
    product_concern[concern_cols].sum(axis=1)
)

print(f"  {'Products in concern summary':<40} "
      f"{len(product_concern):>10,}")
print(f"  {'Concern columns added':<40} "
      f"{len(concern_cols):>10,}")



STEP 7: Build Per-Product Concern Summary

  Products in concern summary                   7,544
  Concern columns added                            14


In [10]:
# ----------------------------------------------------------
# STEP 8: HEADLINE STATISTICS
# ----------------------------------------------------------

print()
print("STEP 8: Headline Statistics")
print()

print("  --- Products affected by each concern type ---")
for c in concern_cols:
    n = (product_concern[c] > 0).sum()
    pct = n / len(product_concern) * 100
    print(f"    {c:<35} {n:>6,} products ({pct:>5.1f}%)")

print()
print("  --- Concern intensity by category ---")
cat_stats = (
    product_concern
        .groupby('primary_category')
        .agg(n_products=('product_id', 'count'),
             avg_concerns=('total_concern_flags', 'mean'),
             pct_with_any=('total_concern_flags',
                          lambda x: (x > 0).mean() * 100))
        .assign(avg_concerns=lambda d: d['avg_concerns'].round(2),
                pct_with_any=lambda d: d['pct_with_any'].round(1))
        .sort_values('avg_concerns', ascending=False)
)
print(cat_stats)



STEP 8: Headline Statistics

  --- Products affected by each concern type ---
    n_annex_iii_restricted               4,805 products ( 63.7%)
    n_carcinogen_concern                   167 products (  2.2%)
    n_drying                             2,228 products ( 29.5%)
    n_endocrine_disruptor                1,412 products ( 18.7%)
    n_environmental_concern                155 products (  2.1%)
    n_eu_banned                            412 products (  5.5%)
    n_fragrance_allergen                 4,450 products ( 59.0%)
    n_irritant                           6,169 products ( 81.8%)
    n_manufacturing_impurity               488 products (  6.5%)
    n_microplastic                          52 products (  0.7%)
    n_nitrosamine_concern                  635 products (  8.4%)
    n_photosensitizer                      388 products (  5.1%)
    n_reproductive_toxin                   485 products (  6.4%)
    n_sensitizer                         3,733 products ( 49.5%)

  --- Conc

In [11]:
# ----------------------------------------------------------
# STEP 9: SAVE OUTPUTS
# ----------------------------------------------------------
 
print()
print("STEP 9: Save Outputs")
print()
 
out_ingredients = os.path.join(OUTPUT_PATH, 'ingredient_tags.csv')
out_long        = os.path.join(OUTPUT_PATH, 'product_ingredient_long_tagged.csv')
out_concern     = os.path.join(OUTPUT_PATH, 'product_concern_summary.csv')
 
ingredient_tags.to_csv(out_ingredients, index=False)
long_tagged.to_csv(out_long, index=False)
product_concern.to_csv(out_concern, index=False)
 
print(f"  Files saved to: {OUTPUT_PATH}")
print()
for file_path in [out_ingredients, out_long, out_concern]:
    print(f"    - {os.path.basename(file_path):<40}")
 
print()



STEP 9: Save Outputs

  Files saved to: ../data/risk_scoring

    - ingredient_tags.csv                     
    - product_ingredient_long_tagged.csv      
    - product_concern_summary.csv             

